[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/068.%20The%20AS_RS%20Task%20Interleaving%20Problem/P68-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 68. The AS/RS Task Interleaving Problem
## Tier 2: A Clarke–Wright Savings Heuristic

### Purpose and inputs

An automated storage and retrieval system (AS/RS) must perform storage and retrieval tasks at specified rack coordinates. This notebook uses a greedy, Clarke–Wright-style savings heuristic to group tasks into capacity-limited routes. The inputs are task identifiers, coordinates, priorities, a depot coordinate, and a maximum number of tasks per route. The outputs are the constructed routes, modeled route times, a sequential baseline, and diagnostic plots.

The method is a heuristic: it ranks task pairs and makes route assignments in that order. It does not guarantee a globally minimum-time plan. A route is an ordered sequence that starts and ends at the depot; each task is assigned to exactly one route.

### Model assumptions

- Coordinates lie on a rectilinear grid, so travel distance is Manhattan distance: the sum of the absolute changes in the two coordinates.
- Travel takes 0.5 seconds per grid-distance unit. A storage operation takes 3 seconds and a retrieval operation takes 2 seconds.
- The depot is the starting and ending point of every route. Tasks within a route are visited in the order in which the heuristic appends them.
- A route can contain at most `route_capacity` tasks. The capacity applies to the whole route, not to each task pair considered during construction.
- Priority and storage–retrieval pairing bonuses affect the heuristic ranking score. They are not physical travel time and are not included in route-time calculations.
- The model omits equipment interference, congestion, task release times, deadlines, precedence constraints, and uncertainty in travel or operation times.

### Pair scores and route times

For tasks $i$ and $j$, the travel-time saving from replacing two separate depot trips with travel between the tasks is

$$s_{ij}^{\mathrm{travel}} = v\left(d_{0i}+d_{0j}-d_{ij}\right),$$

where $0$ denotes the depot, $d_{ab}$ is Manhattan distance between locations $a$ and $b$, and $v=0.5$ seconds per distance unit. The algorithm ranks pairs using the adjusted heuristic score

$$s_{ij}=s_{ij}^{\mathrm{travel}}+0.1(p_i+p_j)+2\,\mathbb{1}_{\{i,j\}\text{ have different task types}},$$

where $p_i$ and $p_j$ are task priorities and the indicator is 1 for a storage–retrieval pair and 0 otherwise. The added terms are ranking bonuses; consequently, $s_{ij}$ is a heuristic score, not a time measurement.

For a route $r=(i_1,\ldots,i_k)$, its modeled time is

$$T(r)=v\left(d_{0i_1}+\sum_{q=1}^{k-1}d_{i_qi_{q+1}}+d_{i_k0}\right)+\sum_{q=1}^{k}o_{i_q},$$

where $o_i=3$ seconds for storage and $o_i=2$ seconds for retrieval. The total plan time is the sum of the route times. The sequential baseline uses the same task order as the input lists—storage tasks followed by retrieval tasks—as one depot-to-depot route. The reported percentage improvement is $(T_{\mathrm{baseline}}-T_{\mathrm{plan}})/T_{\mathrm{baseline}}\times100$; a negative value means the heuristic plan takes longer than this baseline.

### Construction procedure

The algorithm calculates scores for all unordered task pairs, sorts them from highest to lowest, and examines them in that order. If neither task has been assigned, it appends the pair to the first existing route with room for both tasks; otherwise, it starts a new route with the pair. Any task left unassigned after the pair scan becomes a single-task route. Thus, pair assignments are atomic, each task is used once, and capacity is checked against the resulting route length.

For $n$ tasks there are $n(n-1)/2$ pairs. Pair-score calculation and sorting therefore take $O(n^2\log n)$ time; route scans add at most $O(n^2)$ work for this construction. The heuristic is practical for larger instances than exhaustive route enumeration, but its greedy choices do not establish optimality. If scores tie, more than one route plan may be possible; the implementation uses the task-pair order produced by Python's stable sort to resolve such ties.

### What to examine

- Which task pairs receive the largest adjusted scores, and how much of each score is travel saving versus heuristic bonus.
- How the descending-score scan assigns pairs to routes while respecting capacity.
- How route times compare with the sequential baseline. Route operations are included in both quantities, so the comparison is of modeled total time, not travel time alone.
- How changing route capacity affects the greedy plan for this particular instance. A higher capacity does not guarantee a better heuristic result.

The data used in the worked example are the four storage tasks and three retrieval tasks defined below. The plan and performance figures are computed by running the notebook; they are not assumed in advance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# A consistent plotting style makes routes, scores, and performance metrics easier to compare.
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
class ASRSClarkeWrightSavings:
    """Construct AS/RS task routes with a Clarke–Wright-style savings heuristic."""

    def __init__(self, storage_tasks, retrieval_tasks, depot=(1, 1), route_capacity=5):
        """Store the task data and route constraints used by the heuristic.

        Each task is a tuple of (identifier, x, y, priority). The identifier begins
        with 'S' for storage or 'R' for retrieval.
        """
        if not isinstance(route_capacity, int) or route_capacity < 1:
            raise ValueError("route_capacity must be a positive integer")

        self.storage_tasks = list(storage_tasks)
        self.retrieval_tasks = list(retrieval_tasks)
        self.all_tasks = self.storage_tasks + self.retrieval_tasks
        self.depot = depot
        self.route_capacity = route_capacity
        self.n_tasks = len(self.all_tasks)

        task_ids = [task[0] for task in self.all_tasks]
        if len(task_ids) != len(set(task_ids)):
            raise ValueError("Task identifiers must be unique")

    def calculate_distance(self, pos1, pos2):
        """Return rectilinear (Manhattan) distance in grid units."""
        return abs(pos2[0] - pos1[0]) + abs(pos2[1] - pos1[1])

    def calculate_savings(self, task1, task2):
        """Return the adjusted heuristic score for combining two tasks.

        The travel component is measured in seconds. Priority and pairing
        bonuses are score adjustments, not additional seconds.
        """
        pos1 = (task1[1], task1[2])
        pos2 = (task2[1], task2[2])

        depot_to_1 = self.calculate_distance(self.depot, pos1)
        depot_to_2 = self.calculate_distance(self.depot, pos2)
        task_to_task = self.calculate_distance(pos1, pos2)

        travel_saving = 0.5 * (depot_to_1 + depot_to_2 - task_to_task)
        priority_bonus = 0.1 * (task1[3] + task2[3])
        different_types = task1[0].startswith('S') != task2[0].startswith('S')
        pairing_bonus = 2.0 if different_types else 0.0

        return travel_saving + priority_bonus + pairing_bonus

    def calculate_all_savings(self):
        """Score every unordered task pair and sort by descending score."""
        savings_list = []

        for i in range(self.n_tasks):
            for j in range(i + 1, self.n_tasks):
                task1 = self.all_tasks[i]
                task2 = self.all_tasks[j]
                savings = self.calculate_savings(task1, task2)
                savings_list.append((savings, i, j, task1, task2))

        savings_list.sort(key=lambda item: item[0], reverse=True)
        return savings_list

    def calculate_route_time(self, route):
        """Calculate depot-to-depot travel and task-operation time in seconds."""
        if not route:
            return 0.0

        total_time = 0.0
        current_pos = self.depot

        for task in route:
            task_pos = (task[1], task[2])
            total_time += self.calculate_distance(current_pos, task_pos) * 0.5
            total_time += 3.0 if task[0].startswith('S') else 2.0
            current_pos = task_pos

        total_time += self.calculate_distance(current_pos, self.depot) * 0.5
        return total_time

    def build_routes(self):
        """Greedily assign high-scoring task pairs, then add unassigned tasks singly."""
        savings_list = self.calculate_all_savings()
        routes = []
        assigned_tasks = set()

        print("Route Construction Process:")
        print("=" * 50)

        for savings, i, j, task1, task2 in savings_list:
            if i in assigned_tasks or j in assigned_tasks:
                continue

            # A pair must fit as a whole; otherwise the route could exceed its task limit.
            if self.route_capacity < 2:
                continue

            target_route = next(
                (route for route in routes
                 if len(route) + 2 <= self.route_capacity),
                None,
            )

            if target_route is None:
                target_route = []
                routes.append(target_route)
                action = "Created new route"
            else:
                action = "Added pair to route"

            target_route.extend([task1, task2])
            assigned_tasks.update((i, j))
            route_time = self.calculate_route_time(target_route)
            task_names = [task[0] for task in target_route]
            print(
                f"{action}: {task1[0]} + {task2[0]} -> {task_names} "
                f"(Time: {route_time:.1f} s, Score: {savings:.1f})"
            )

        for i, task in enumerate(self.all_tasks):
            if i not in assigned_tasks:
                routes.append([task])
                route_time = self.calculate_route_time([task])
                print(f"Added single-task route: {task[0]} (Time: {route_time:.1f} s)")

        return routes

    def calculate_sequential_baseline(self):
        """Calculate time for one route in input order: storage, then retrieval."""
        sequential_route = self.storage_tasks + self.retrieval_tasks
        return self.calculate_route_time(sequential_route)

    def analyze_solution(self, routes):
        """Report route times and compare their sum with the sequential baseline."""
        print("\nSolution Analysis:")
        print("=" * 50)

        route_times = []
        total_system_time = 0.0
        routed_task_ids = []

        for i, route in enumerate(routes):
            route_time = self.calculate_route_time(route)
            route_times.append(route_time)
            total_system_time += route_time
            routed_task_ids.extend(task[0] for task in route)
            task_names = [task[0] for task in route]
            print(f"Route {i + 1}: {task_names} - Time: {route_time:.2f} seconds")

        expected_task_ids = [task[0] for task in self.all_tasks]
        assert len(routed_task_ids) == len(expected_task_ids)
        assert set(routed_task_ids) == set(expected_task_ids)
        assert len(routed_task_ids) == len(set(routed_task_ids))
        assert all(len(route) <= self.route_capacity for route in routes)
        assert np.isclose(sum(route_times), total_system_time)

        sequential_time = self.calculate_sequential_baseline()
        if np.isclose(sequential_time, 0.0):
            improvement = 0.0
        else:
            improvement = (
                (sequential_time - total_system_time) / sequential_time * 100
            )

        print(f"\nTotal System Time: {total_system_time:.2f} seconds")
        print(f"Sequential Baseline: {sequential_time:.2f} seconds")
        print(f"Change relative to baseline: {improvement:.1f}%")

        return {
            'route_times': route_times,
            'total_time': total_system_time,
            'sequential_time': sequential_time,
            'improvement_percentage': improvement,
            'num_routes': len(routes),
        }

In [ ]:
# Define the worked example as (identifier, x-coordinate, y-coordinate, priority).
storage_tasks = [
    ('S1', 2, 3, 5),
    ('S2', 6, 5, 3),
    ('S3', 8, 2, 4),
    ('S4', 4, 6, 2),
]

retrieval_tasks = [
    ('R1', 3, 4, 8),
    ('R2', 7, 3, 6),
    ('R3', 5, 7, 2),
]

cw_solver = ASRSClarkeWrightSavings(
    storage_tasks,
    retrieval_tasks,
    route_capacity=5,
)

print("AS/RS Task Interleaving - Clarke–Wright Savings Heuristic")
print(f"Storage tasks: {len(storage_tasks)}")
print(f"Retrieval tasks: {len(retrieval_tasks)}")
print(f"Total tasks: {cw_solver.n_tasks}")
print(f"Route capacity: {cw_solver.route_capacity}")
print(f"Depot position: {cw_solver.depot}")
print()

print("Task Details:")
for task in cw_solver.all_tasks:
    task_type = "Storage" if task[0].startswith('S') else "Retrieval"
    print(
        f"  {task[0]}: {task_type} at ({task[1]}, {task[2]}), "
        f"priority={task[3]}"
    )

In [ ]:
# Inspect the ranked pair scores before the greedy route-construction step.
print("\nTask Pair Score Analysis:")
print("=" * 60)

savings_list = cw_solver.calculate_all_savings()
print("Top 10 task pairs:")

for i, (score, idx1, idx2, task1, task2) in enumerate(savings_list[:10]):
    pair_type = (
        "Storage–retrieval"
        if task1[0].startswith('S') != task2[0].startswith('S')
        else "Same task type"
    )
    print(
        f"{i + 1:2d}. {task1[0]} + {task2[0]}: "
        f"adjusted score = {score:6.2f} ({pair_type})"
    )

print(f"\nTotal possible pairs: {len(savings_list)}")

In [ ]:
# Construct routes in descending pair-score order, then evaluate the complete plan.
print("\n" + "=" * 60)
print("CLARKE–WRIGHT-STYLE ROUTE CONSTRUCTION")
print("=" * 60)

routes = cw_solver.build_routes()
analysis_results = cw_solver.analyze_solution(routes)

# Confirm the reported plan total equals the sum of its route times.
assert np.isclose(
    analysis_results['total_time'],
    sum(analysis_results['route_times']),
)

In [ ]:
from matplotlib.patches import Patch

# The four panels show the constructed routes, pair scores, route times, and baseline comparison.
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
route_colors = plt.cm.Set3(np.linspace(0, 1, max(len(routes), 1)))

# Route map: marker shape identifies task type; line color identifies the route.
for i, route in enumerate(routes):
    route_color = route_colors[i]

    for task in route:
        marker = 's' if task[0].startswith('S') else 'o'
        ax1.scatter(
            task[1], task[2], color=route_color, s=100,
            marker=marker, alpha=0.85,
        )
        ax1.annotate(
            task[0], (task[1], task[2]), xytext=(5, 5),
            textcoords='offset points', fontsize=10,
        )

    route_x = [cw_solver.depot[0]] + [task[1] for task in route]
    route_y = [cw_solver.depot[1]] + [task[2] for task in route]
    route_x.append(cw_solver.depot[0])
    route_y.append(cw_solver.depot[1])
    ax1.plot(
        route_x, route_y, color=route_color, alpha=0.7,
        linewidth=2, label=f"Route {i + 1}",
    )

ax1.scatter(
    cw_solver.depot[0], cw_solver.depot[1], color='red',
    s=200, marker='*', label='Depot', zorder=5,
)
ax1.set_xlabel("X coordinate (grid units)")
ax1.set_ylabel("Y coordinate (grid units)")
ax1.set_title("AS/RS Task Routes")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Score chart: orange bars denote storage–retrieval pairs; blue bars denote same-type pairs.
top_pairs = savings_list[:20]
score_values = [item[0] for item in top_pairs]
pair_labels = [f"{item[3][0]} + {item[4][0]}" for item in top_pairs]
bar_colors = [
    'orange'
    if item[3][0].startswith('S') != item[4][0].startswith('S')
    else 'steelblue'
    for item in top_pairs
]
ax2.bar(range(len(score_values)), score_values, color=bar_colors, alpha=0.8)
ax2.set_xlabel("Task pairs (ranked, top 20)")
ax2.set_ylabel("Adjusted heuristic score (score units)")
ax2.set_title("Task Pair Scores")
ax2.set_xticks(range(len(score_values)))
ax2.set_xticklabels(pair_labels, rotation=45, ha='right')
ax2.legend(handles=[
    Patch(color='orange', label='Storage–retrieval pair'),
    Patch(color='steelblue', label='Same task type'),
])
ax2.grid(True, axis='y', alpha=0.3)

# Route-time chart uses the same route colors as the map.
route_names = [f"Route {i + 1}" for i in range(len(routes))]
route_times = analysis_results['route_times']
bars3 = ax3.bar(route_names, route_times, color=route_colors, alpha=0.8)
ax3.set_xlabel("Route")
ax3.set_ylabel("Modeled route time (seconds)")
ax3.set_title("Individual Route Times")
ax3.grid(True, axis='y', alpha=0.3)
for bar, route_time in zip(bars3, route_times):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{route_time:.1f} s",
        ha='center',
        va='bottom',
    )

# The plan and baseline use the same modeled time components.
methods = ['Sequential baseline', 'Heuristic plan']
times = [
    analysis_results['sequential_time'],
    analysis_results['total_time'],
]
bars4 = ax4.bar(methods, times, color=['firebrick', 'seagreen'], alpha=0.8)
ax4.set_ylabel("Modeled total time (seconds)")
ax4.set_title(
    f"Plan vs. Baseline ({analysis_results['improvement_percentage']:.1f}% change)"
)
ax4.grid(True, axis='y', alpha=0.3)
for bar, total_time in zip(bars4, times):
    ax4.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        f"{total_time:.1f} s",
        ha='center',
        va='bottom',
    )

plt.tight_layout()
plt.show()

In [ ]:
# Rebuild the same instance under several route-capacity limits.
print("\n" + "=" * 60)
print("SENSITIVITY ANALYSIS: ROUTE CAPACITY")
print("=" * 60)

capacities = [2, 3, 4, 5, 6, 7]
capacity_results = []

for capacity in capacities:
    solver_temp = ASRSClarkeWrightSavings(
        storage_tasks,
        retrieval_tasks,
        route_capacity=capacity,
    )
    routes_temp = solver_temp.build_routes()
    analysis_temp = solver_temp.analyze_solution(routes_temp)

    capacity_results.append({
        'capacity': capacity,
        'num_routes': analysis_temp['num_routes'],
        'total_time': analysis_temp['total_time'],
        'improvement': analysis_temp['improvement_percentage'],
    })

    print(
        f"Capacity {capacity}: {analysis_temp['num_routes']} routes, "
        f"{analysis_temp['total_time']:.2f} s, "
        f"{analysis_temp['improvement_percentage']:.1f}% change"
    )

# Compare route counts and time changes across the tested capacities.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
cap_values = [result['capacity'] for result in capacity_results]
num_routes = [result['num_routes'] for result in capacity_results]
improvements = [result['improvement'] for result in capacity_results]
total_times = [result['total_time'] for result in capacity_results]

ax1.plot(cap_values, num_routes, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel("Maximum tasks per route")
ax1.set_ylabel("Number of routes")
ax1.set_title("Route Count vs. Capacity")
ax1.grid(True, alpha=0.3)
ax1.set_xticks(capacities)

ax2_twin = ax2.twinx()
line1 = ax2.plot(
    cap_values, improvements, 'go-', linewidth=2,
    markersize=8, label='Change vs. baseline (%)',
)
line2 = ax2_twin.plot(
    cap_values, total_times, 'r--', linewidth=2,
    marker='o', label='Plan time (seconds)',
)
ax2.set_xlabel("Maximum tasks per route")
ax2.set_ylabel("Change vs. baseline (%)", color='green')
ax2_twin.set_ylabel("Plan time (seconds)", color='red')
ax2.set_title("Plan Performance vs. Capacity")
ax2.grid(True, alpha=0.3)
ax2.set_xticks(capacities)
lines = line1 + line2
ax2.legend(lines, [line.get_label() for line in lines], loc='best')

plt.tight_layout()
plt.show()

# This identifies the best result among the capacities tested, not a universal optimum.
best_result = max(capacity_results, key=lambda result: result['improvement'])
print(
    f"\nBest tested capacity: {best_result['capacity']} "
    f"({best_result['improvement']:.1f}% change vs. baseline)"
)

In [ ]:
import time

print("\n" + "=" * 60)
print("SCALABILITY EXPERIMENT")
print("=" * 60)

def generate_random_tasks(n_storage, n_retrieval, grid_size=20):
    """Generate random task coordinates and priorities for a size experiment."""
    storage = [
        (
            f'S{i}',
            np.random.randint(1, grid_size),
            np.random.randint(1, grid_size),
            np.random.randint(1, 10),
        )
        for i in range(n_storage)
    ]
    retrieval = [
        (
            f'R{i}',
            np.random.randint(1, grid_size),
            np.random.randint(1, grid_size),
            np.random.randint(1, 10),
        )
        for i in range(n_retrieval)
    ]
    return storage, retrieval

# Random instances are generated without a fixed seed, so their results can vary between runs.
problem_sizes = [(4, 3), (6, 4), (8, 6), (10, 8), (12, 10)]
scalability_results = []

for n_storage, n_retrieval in problem_sizes:
    storage_test, retrieval_test = generate_random_tasks(
        n_storage,
        n_retrieval,
    )

    start_time = time.perf_counter()
    solver_test = ASRSClarkeWrightSavings(storage_test, retrieval_test)
    routes_test = solver_test.build_routes()
    analysis_test = solver_test.analyze_solution(routes_test)
    computation_time = time.perf_counter() - start_time

    scalability_results.append({
        'total_tasks': n_storage + n_retrieval,
        'computation_time': computation_time,
        'improvement': analysis_test['improvement_percentage'],
        'num_routes': analysis_test['num_routes'],
    })

    print(
        f"Size {n_storage + n_retrieval:2d} tasks: "
        f"{computation_time:.4f} s, "
        f"{analysis_test['improvement_percentage']:.1f}% change, "
        f"{analysis_test['num_routes']} routes"
    )

# Plot computation time, observed plan change, and route count for these instances.
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
sizes = [result['total_tasks'] for result in scalability_results]
times = [result['computation_time'] for result in scalability_results]
improvements = [result['improvement'] for result in scalability_results]
route_counts = [result['num_routes'] for result in scalability_results]

ax1.plot(sizes, times, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel("Problem size (tasks)")
ax1.set_ylabel("Computation time (seconds)")
ax1.set_title("Observed Computation Time")
ax1.grid(True, alpha=0.3)
if all(computation_time > 0 for computation_time in times):
    ax1.set_yscale('log')

ax2.plot(sizes, improvements, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel("Problem size (tasks)")
ax2.set_ylabel("Change vs. baseline (%)")
ax2.set_title("Observed Plan Change")
ax2.grid(True, alpha=0.3)

ax3.plot(sizes, route_counts, 'ro-', linewidth=2, markersize=8)
ax3.set_xlabel("Problem size (tasks)")
ax3.set_ylabel("Number of routes")
ax3.set_title("Route Count")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Runtime ratios are descriptive measurements, not proofs of asymptotic complexity.
print("\nObserved size and runtime ratios:")
for previous, current in zip(scalability_results, scalability_results[1:]):
    size_ratio = current['total_tasks'] / previous['total_tasks']
    if previous['computation_time'] > 0:
        time_ratio = current['computation_time'] / previous['computation_time']
        time_ratio_text = f"{time_ratio:.2f}"
    else:
        time_ratio_text = "undefined (previous runtime was zero)"
    print(
        f"{previous['total_tasks']} to {current['total_tasks']} tasks: "
        f"size ratio = {size_ratio:.2f}, runtime ratio = {time_ratio_text}"
    )

## Computational role, strengths, and limitations

The savings heuristic replaces exhaustive route-plan enumeration with a ranked list of task pairs and a greedy assignment pass. For $n$ tasks, it forms $n(n-1)/2$ pair scores and sorts them, giving an overall $O(n^2\log n)$ time bound for the implementation described here. The route-construction scan and the calculation of route times do not increase that asymptotic bound. The scalability experiment measures actual runtimes on randomly generated instances; because no random seed is fixed, its task data and observed measurements may differ between runs.

The method is transparent: each candidate pair has an explicit score, and routes can be traced through the descending-score scan. Its greedy nature also limits what can be concluded: a high-ranked pair is considered early, assignments are not subsequently rearranged, and no global optimality guarantee is provided. The priority and task-type bonuses can influence pair rankings even when another pair would yield a better total modeled route time. Capacity sensitivity is therefore an empirical property of the tested instance, not a general rule that larger capacity improves performance.

### Scope and interpretation

This notebook demonstrates a single-route-list construction heuristic for a simplified AS/RS model. It does not represent equipment scheduling, interference, deadlines, or operational uncertainty. The sequential baseline is a reference plan, not a proven lower bound; a positive or negative percentage change should be interpreted only relative to that baseline and the stated time assumptions. The plots and printed metrics describe the plan produced by the heuristic, rather than an optimality certificate.